In [4]:
import json
import math
from collections import Counter
import pandas as pd

TEST_JSON = r"E:\Projects\CheckThat 2026\task 2\data\test.json"

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)


def entropy(labels):
    """
    Shannon entropy of Verdict_list.
    Higher entropy = more disagreement.
    """
    counts = Counter(labels)
    total = len(labels)

    ent = 0.0
    for c in counts.values():
        p = c / total
        ent -= p * math.log2(p)

    return ent


def interestingness(example):

    score = 0

    # -----------------------------
    # 1. Disagreement (MOST IMPORTANT)
    # -----------------------------
    H = entropy(example["Verdict_list"])
    score += H * 100

    # -----------------------------
    # 2. 20 traces preferred
    # -----------------------------
    score += len(example["Reasoning_traces"]) * 5

    # -----------------------------
    # 3. More evidences preferred
    # -----------------------------
    score += len(example["evidences"]) * 4

    # -----------------------------
    # 4. Longer claim
    # -----------------------------
    score += len(example["claim"].split()) * 0.5

    # -----------------------------
    # 5. Longer reasoning traces
    # -----------------------------
    avg_trace_len = sum(
        len(t.split())
        for t in example["Reasoning_traces"]
    ) / len(example["Reasoning_traces"])

    score += avg_trace_len * 0.2

    # -----------------------------
    # 6. Final verdict differs from majority
    # -----------------------------
    counts = Counter(example["Verdict_list"])
    majority = counts.most_common(1)[0][0]

    if majority != example["verdict"]:
        score += 40

    return score, H, counts


results = []

for ex in test_data:

    score, H, counts = interestingness(ex)

    results.append({
        "query_id": ex["query_id"],
        "Label": ex["Label"],
        "Final Verdict": ex["verdict"],
        "Entropy": round(H,3),
        "Counts": dict(counts),
        "Interestingness": round(score,2),
        "Example": ex
    })


results = sorted(
    results,
    key=lambda x: x["Interestingness"],
    reverse=True
)


summary = pd.DataFrame([

    {
        "query_id": r["query_id"],
        "Label": r["Label"],
        "Final Verdict": r["Final Verdict"],
        "Entropy": r["Entropy"],
        "Interestingness": r["Interestingness"],
        "Verdict Counts": str(r["Counts"])
    }

    for r in results

])

display(summary.head(20))

,query_id,Label,Final Verdict,Entropy,Interestingness,Verdict Counts
0,925,False,Conflicting,1.559,346.52,"{'True': 7, 'Conflicting': 8, 'False': 5}"
1,179,False,Conflicting,1.500,338.57,"{'False': 5, 'Conflicting': 10, 'True': 5}"
2,1605,True,Conflicting,1.559,337.74,"{'True': 5, 'Conflicting': 8, 'False': 7}"
3,2260,True,True,1.219,332.40,"{'True': 11, 'False': 1, 'Conflicting': 8}"
4,2413,False,Conflicting,1.581,331.99,"{'Conflicting': 7, 'True': 7, 'False': 6}"
5,1031,False,False,1.353,329.27,"{'False': 12, 'True': 5, 'Conflicting': 3}"
6,1432,True,False,1.458,329.24,"{'False': 9, 'Conflicting': 8, 'True': 3}"
7,1209,True,Conflicting,1.458,327.66,"{'Conflicting': 9, 'True': 8, 'False': 3}"
8,2171,True,Conflicting,1.181,327.49,"{'Conflicting': 14, 'False': 3, 'True': 3}"
9,156,True,True,1.441,327.08,"{'False': 3, 'True': 10, 'Conflicting': 7}"


In [5]:
best = results[0]["Example"]

print("="*120)
print("MOST INTERESTING EXAMPLE")
print("="*120)

print(f"\nQuery ID : {best['query_id']}")
print(f"Golden Label : {best['Label']}")
print(f"Final Verdict : {best['verdict']}")

print("\nCLAIM")
print("-"*120)
print(best["claim"])

print("\nEVIDENCES")
print("-"*120)

for i, e in enumerate(best["evidences"],1):
    print(f"{i}. {e}")

print("\nREASONING TRACES")
print("-"*120)

for i,(trace,v) in enumerate(zip(best["Reasoning_traces"],
                                 best["Verdict_list"]),1):

    print(f"\nTrace {i}")
    print(f"Verdict : {v}")
    print(trace)

MOST INTERESTING EXAMPLE

Query ID : 925
Golden Label : False
Final Verdict : Conflicting

CLAIM
------------------------------------------------------------------------------------------------------------------------
In August 2025, Florida Sen. Rick Scott disclosed more than $26 million in stock trades more than a year later than the deadline required by law.

EVIDENCES
------------------------------------------------------------------------------------------------------------------------
1. Beginning today, members and senior staff1of the U.S. House of Representatives and U.S. Senate are required to disclose certain financial transactions no later than 45 days after the transaction has occurred. This new disclosure requirement is a result of the enactment of the Stop Trading on Congressional Knowledge Act (STOCK Act), signed into law on April 4, 2012. In addition to this new disclosure requirement, the STOCK Act: You must not provide any material, non-public information to Congress 

In [6]:
for rank, r in enumerate(results[:5], start=1):

    ex = r["Example"]

    print("="*120)
    print(f"RANK {rank}")
    print("="*120)

    print(f"Query ID : {ex['query_id']}")
    print(f"Interestingness : {r['Interestingness']}")
    print(f"Entropy : {r['Entropy']:.3f}")
    print(f"Verdict Counts : {r['Counts']}")
    print(f"Golden Label : {ex['Label']}")
    print(f"Final Verdict : {ex['verdict']}")
    print(f"Claim : {ex['claim']}")
    print()

RANK 1
Query ID : 925
Interestingness : 346.52
Entropy : 1.559
Verdict Counts : {'True': 7, 'Conflicting': 8, 'False': 5}
Golden Label : False
Final Verdict : Conflicting
Claim : In August 2025, Florida Sen. Rick Scott disclosed more than $26 million in stock trades more than a year later than the deadline required by law.

RANK 2
Query ID : 179
Interestingness : 338.57
Entropy : 1.500
Verdict Counts : {'False': 5, 'Conflicting': 10, 'True': 5}
Golden Label : False
Final Verdict : Conflicting
Claim : "Tea production has grown, with earnings rising from KSh138 billion in 2022 to KSh215 billion in 2024, a 56% increase."

RANK 3
Query ID : 1605
Interestingness : 337.74
Entropy : 1.559
Verdict Counts : {'True': 5, 'Conflicting': 8, 'False': 7}
Golden Label : True
Final Verdict : Conflicting
Claim : In March 2026, the FBI warned law enforcement agencies in California of a potential threat from Iranian drone strikes.

RANK 4
Query ID : 2260
Interestingness : 332.4
Entropy : 1.219
Verdict Cou